### Random vs Random

In [2]:
from collections import Counter, defaultdict
from domain.configs import MAX_STEPS_PER_EPISODE
from environment.grenight_environment import GrenightEnvironment

In [3]:
def play_random_game(env_arg: GrenightEnvironment) -> tuple[str, dict, int]:

    env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        action = env_arg.sample()
        _, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", info, move_count
    if reward == 0.0:
        return "draw", info, move_count

    winner_is_white = acting_player_is_white if reward == 1.0 else not acting_player_is_white

    return ("white_win", info, move_count) if winner_is_white else ("black_win", info, move_count)

In [4]:
env = GrenightEnvironment()
outcomes_counter = Counter()
draw_reasons_counter = Counter()
total_moves_per_outcome = defaultdict(int)

for _ in range(1_000):
    outcome, game_info, move_count = play_random_game(env)
    outcomes_counter[outcome] += 1
    total_moves_per_outcome[outcome] += move_count
    if game_info["draw_reason"] is not None:
        draw_reasons_counter[game_info["draw_reason"]] += 1

average_moves_per_outcome = dict()
for outcome, total_moves in total_moves_per_outcome.items():
    average_moves_per_outcome[outcome] = f"{(total_moves / outcomes_counter[outcome]):.2f}"

print(f"STATS OUT FROM: {1_000} GAMES\n"
      f"Outcomes: {outcomes_counter}\n"
      f"Average moves per outcome: {average_moves_per_outcome}\n"
      f"Total average moves: {sum(total_moves_per_outcome.values()) / 1000}\n"
      f"Draw reasons: {draw_reasons_counter}\n")

STATS OUT FROM: 1000 GAMES
Outcomes: Counter({'draw': 646, 'black_win': 197, 'white_win': 157})
Average moves per outcome: {'white_win': '29.30', 'draw': '70.62', 'black_win': '35.25'}
Total average moves: 57.166
Draw reasons: Counter({'insufficient_material': 294, 'stalemate': 233, 'threefold_repetition': 72, 'max_steps_without_progress': 47})

